In [3]:
!pip install catboost

import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')


# 1. YÜKLEME VE SÜTUN TEMİZLİĞİ
train = pd.read_csv('train.csv')
test = pd.read_csv('test_x.csv')


best_submission_file = 'Takim_141_version_6.csv'
pseudo_labels = pd.read_csv(best_submission_file)

train.columns = train.columns.str.lower().str.replace(' ', '_').str.strip()
test.columns = test.columns.str.lower().str.replace(' ', '_').str.strip()
target_col = 'bilissel_performans_skoru'

# 2. TEST VERİSİNE "SAHTE" (AMA ÇOK İYİ) CEVAPLARI EKLEME
test_pseudo = test.copy()
test_pseudo[target_col] = pseudo_labels[target_col] # 1.204'lük tahminleri gerçekmiş gibi ekliyoruz

# EĞİTİM VE TEST VERİSİNİ BİRLEŞTİR (Devasa bir eğitim seti yaratıyoruz)
combined_train = pd.concat([train, test_pseudo], axis=0).reset_index(drop=True)

print(f"Orijinal Eğitim Verisi: {len(train)} satır")
print(f"Pseudo-Label Eklenmiş Yeni Eğitim Verisi: {len(combined_train)} satır\n")

# 3. AYNI GÜVENLİ ÖZELLİKLER
def create_safe_features(df):
    df = df.copy()
    if 'rem_yuzdesi' in df.columns and 'derin_uyku_yuzdesi' in df.columns:
        df['toplam_kaliteli_uyku'] = df['rem_yuzdesi'] + df['derin_uyku_yuzdesi']
    if 'uyku_oncesi_kafein_mg' in df.columns and 'uyku_oncesi_ekran_suresi_dk' in df.columns:
        df['uyku_bozucu_faktorler'] = (df['uyku_oncesi_kafein_mg']/100) + (df['uyku_oncesi_ekran_suresi_dk']/30)
    if 'gunluk_calisma_saati' in df.columns and 'stres_skoru' in df.columns:
        df['tukenmislik_riski'] = df['gunluk_calisma_saati'] * df['stres_skoru']
    return df

train = create_safe_features(train) # Apply feature engineering to original train DataFrame
combined_train = create_safe_features(combined_train)
test = create_safe_features(test)

y_combined = combined_train[target_col]
X_combined = combined_train.drop(['id', target_col], axis=1)
X_test = test.drop(['id'], axis=1)

# 4. EKSİK VERİ DOLDURMA (Native CatBoost formatı)
num_cols = X_combined.select_dtypes(include=[np.number]).columns
for col in num_cols:
    med_val = train[col].median() # Medyanı sadece orijinal train'den alıyoruz (Leakage önlemi)
    X_combined[col] = X_combined[col].fillna(med_val)
    X_test[col] = X_test[col].fillna(med_val)

cat_cols = X_combined.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    X_combined[col] = X_combined[col].fillna('Missing').astype(str)
    X_test[col] = X_test[col].fillna('Missing').astype(str)

# 5. MULTI-SEED FULL FIT (Artık Validation yok, tüm veriyi ezberliyoruz!)
SEEDS = [42, 123, 777, 2024, 9999] # 5 farklı evren
test_predictions = np.zeros(len(X_test))

print("Pseudo-Label verisiyle 5 farklı model eğitiliyor...")

for i, seed in enumerate(SEEDS):
    print(f" -> Model {i+1}/5 (Seed: {seed}) eğitiliyor...")
    model = CatBoostRegressor(
        iterations=2000,
        learning_rate=0.015,
        depth=6,
        l2_leaf_reg=5,
        cat_features=cat_cols,
        random_seed=seed,
        verbose=0
    )

    # Tüm birleşik veriyle eğitiyoruz
    model.fit(X_combined, y_combined)
    test_predictions += model.predict(X_test) / len(SEEDS)

print("\n" + "="*60)
print("PSEUDO-LABELING TAMAMLANDI!")
print("="*60)

# Skorları 0-10 Sınırlarında Tutma
test_predictions = np.clip(test_predictions, 0, 10)
submission = pd.DataFrame({'id': test['id'], target_col: test_predictions})
submission.to_csv('Takim_141_version_7.csv', index=False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.5 MB/s eta 0:00:00
Orijinal Eğitim Verisi: 56000 satır
Pseudo-Label Eklenmiş Yeni Eğitim Verisi: 80000 satır

Pseudo-Label verisiyle 5 farklı model eğitiliyor...
 -> Model 1/5 (Seed: 42) eğitiliyor...
 -> Model 2/5 (Seed: 123) eğitiliyor...
 -> Model 3/5 (Seed: 777) eğitiliyor...
 -> Model 4/5 (Seed: 2024) eğitiliyor...
 -> Model 5/5 (Seed: 9999) eğitiliyor...

PSEUDO-LABELING TAMAMLANDI!
